## Open government data provided by **opendata.swiss**

_Automatically generated Python starter code for dataset_ **38b38577-b42a-4eb8-99e6-8b18bd77c4bf**


## Dataset

# Bilan démographique de la population résidante permanente, par commune, sexe et catégorie de nationalité, dès 1981


## Description

Ce jeu de données présente le bilan démographique de la population du canton de Fribourg, ventilé par genre, catégorie de nationalité et niveau géographique &#40;district, commune&#41;, dès 1981.Le bilan démographique décrit, pour chaque année civile, la variation de l&#x27;effectif de la population résultant des mouvements naturels &#40;naissances, décès&#41; et spatiaux &#40;immigrations et émigrations internationales, arrivées et départs intercantonaux et intracantonaux&#41;, ainsi que des changements de type de population, des changements de nationalité, des changements de genre et d&#x27;éventuelles corrections statistiques. Il comprend notamment la population au 1er janvier, la population au 31 décembre et l&#x27;accroissement total. La population de référence est la population résidante permanente. Jusqu&#x27;en 2010, celle-ci comprend les personnes de nationalité suisse ayant leur domicile principal en Suisse ainsi que les ressortissants étrangers titulaires d&#x27;une autorisation de séjour valable au moins 12 mois. Depuis le 1er janvier 2011, avec l&#x27;introduction de STATPOP, elle inclut en outre les personnes en procédure d&#x27;asile séjournant en Suisse depuis au moins 12 mois. Des écarts ponctuels entre l&#x27;état de la population au 31 décembre et celui du 1er janvier de l&#x27;année suivante peuvent apparaître, notamment lors des adaptations aux recensements fédéraux de la population &#40;1990/1991 et 2000/2001&#41;, du passage d&#x27;ESPOP à STATPOP &#40;2010/2011&#41; ou de modifications territoriales.Les données sont établies par l&#x27;Office fédéral de la statistique. Elles reposent sur la Statistique de l&#x27;état annuel de la population &#40;ESPOP&#41; pour la période 1981-2010, puis sur la Statistique de la population et des ménages &#40;STATPOP&#41; dès 2011.


## Dataset links

[View this dataset in the data catalogue](https://opendata.swiss/de/dataset/bevolkerungsbilanz-der-standigen-wohnbevolkerung-nach-gemeinde-geschlecht-und-staatsangeho-1981)


## Metadata

- **identifier:** 01&#95;03&#95;bilan&#95;demo&#95;sex&#95;natio@ssd
- **publisher.name:** ssd
- **organization.display&#95;name:** Amt für Statistik und Daten Freiburg
- **organization.url:** https://www.fr.ch/deef/ssd
- **maintainer:** 
- **maintainer&#95;email:** 
- **keywords:** commune, nationalite, genre, population-residante
- **issued:** 2026-08-20 00:00:00
- **metadata&#95;created:** 2025-01-30T19:23:16.725355
- **metadata&#95;modified:** 2026-09-15T02:07:05.133436
- **Resource id:** d445b421-26c7-4eb9-bf87-9258295ebb5f
- **Resource name:** csv
- **Resource package&#95;id:** 38b38577-b42a-4eb8-99e6-8b18bd77c4bf
- **Resource description:** Bilan démographique de la population résidante permanente, par commune, sexe et catégorie de nationalité, dès 1981 &#40;csv&#41;
- **Resource issued:** 2026-08-20 00:00:00
- **Resource modified:** 2026-08-20 07:27:57.066000+00:00
- **Resource license:** https://opendata.swiss/terms-of-use&#35;terms&#95;by
- **Resource rights:** http://www.opendefinition.org/licenses/cc-by-sa


## Imports and helper functions

This template targets pandas (3.x), seaborn (0.13+), matplotlib (3.x), and requests (2.x); packages are not installed or pinned by the notebook. Check the versions below when sharing results.


In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

sns.set_theme(style="whitegrid")
print(f"pandas {pd.__version__}; seaborn {sns.__version__}")

In [ ]:
from io import BufferedReader
from typing import BinaryIO

import requests

USER_AGENT = "OpendataSwiss-StarterCode-Python/1.0"
HTTP_TIMEOUT = 30


def get_dataset(
    url: str | BinaryIO,
    *,
    sep: str | None = None,
    drop_empty_unnamed: bool = True,
    **kwargs: object,
) -> pd.DataFrame:
    """Read one CSV resource; preserve data and report parsing failures."""
    if isinstance(url, str) and url.startswith(("https://", "http://")):
        with requests.get(
            url, headers={"User-Agent": USER_AGENT}, timeout=HTTP_TIMEOUT, stream=True
        ) as response:
            response.raise_for_status()
            # Decode HTTP compression while streaming CSV bytes into pandas.
            response.raw.decode_content = True
            # Let the buffer finish reading after the HTTP connection reaches EOF.
            response.raw.auto_close = False
            with BufferedReader(response.raw) as source:
                # Detect file compression after HTTP compression has been decoded.
                if kwargs.get("compression", "infer") == "infer" and source.peek(2).startswith(
                    b"\x1f\x8b"
                ):
                    kwargs["compression"] = "gzip"
                return get_dataset(source, sep=sep, drop_empty_unnamed=drop_empty_unnamed, **kwargs)
    data = pd.read_csv(url, sep=sep, engine="python", dtype_backend="numpy_nullable", **kwargs)
    # Extra fields can make pandas silently consume leading values as an index.
    if kwargs.get("index_col") is None and not isinstance(data.index, pd.RangeIndex):
        raise pd.errors.ParserError(
            "CSV rows contain unexpected index fields. Check the delimiter and header; "
            "set index_col explicitly only if these fields are an intentional index."
        )
    if drop_empty_unnamed and len(data) > 0:
        # Blank CSV headers (often from trailing separators) become "Unnamed: n".
        unnamed = data.columns.astype(str).str.fullmatch(r"Unnamed: \d+(?:\.\d+)?")
        empty_unnamed = unnamed & data.isna().all()
        if empty_unnamed.any():
            print("Removed empty unnamed columns:", data.columns[empty_unnamed].tolist())
            data = data.loc[:, ~empty_unnamed]
    return data

## Load data

Selected resource: **csv** (CSV). This notebook loads only this resource.

For CSV, the separator is guessed. If parsing looks wrong (especially for a single-column file), set `sep=","` or `sep=";"` in `get_dataset(...)`. Check inferred types, missing-value codes, decimal marks, and identifiers with leading zeros; use `dtype`, `na_values`, `decimal`, or `encoding` as needed.

Unexpected index inference raises a parsing error to prevent values shifting under the wrong headers. Check the delimiter and header, or set `index_col` explicitly for an intentional index.

Fully empty columns named `Unnamed: n` are removed on import. Named empty columns and unnamed columns containing values are kept. Use `drop_empty_unnamed=False` to retain every column, including intentionally named `Unnamed: n` fields.


In [ ]:
df = get_dataset("https://opendata.fr.ch/api/explore/v2.1/catalog/datasets/01_03_bilan_demo_sex_natio/exports/csv?use_labels=true")

## Analyze data

Inspect before cleaning: these cells leave `df` unchanged. Duplicate rows and missing values may be meaningful; check the metadata before removing or filling them.


In [ ]:
print(f"{len(df):,} rows × {len(df.columns):,} columns")
print(f"{df.duplicated().sum():,} duplicate rows after the first occurrence")
df.info()
display(df.head())

In [ ]:
# Column types, completeness, and distinct non-missing values.
profile = pd.DataFrame(
    {
        "dtype": df.dtypes.astype(str),
        "missing": df.isna().sum(),
        "missing_pct": df.isna().mean().mul(100),
        "unique": df.nunique(dropna=True),
    }
).sort_values("missing_pct", ascending=False)
display(profile.round(1))

In [ ]:
# Show up to 20 columns with missing values; percentages use all rows.
missing = profile.loc[profile["missing"] > 0].head(20).rename_axis("column").reset_index()
if not missing.empty:
    fig, ax = plt.subplots(figsize=(8, max(2, len(missing) * 0.3)))
    sns.barplot(data=missing, x="missing_pct", y="column", color="steelblue", errorbar=None, ax=ax)
    ax.set(xlim=(0, 100), xlabel="Missing values (%)", ylabel="", title="Missingness by column")
    fig.tight_layout()
    plt.show()
else:
    print("No missing values to plot (or the dataset has no rows).")

In [ ]:
# Summarize numeric and non-numeric columns, including pandas string types.
if not df.empty:
    display(df.describe(include="all").T)
else:
    print("No data to summarize.")

In [ ]:
# Plot up to four numeric columns. Exclude missing/infinite values only from plots.
numeric = df.select_dtypes(include="number")
for column in numeric.columns[:4]:
    values = numeric[column].dropna()
    values = values.loc[~values.isin([float("inf"), -float("inf")])]
    if not values.empty:
        fig, ax = plt.subplots(figsize=(7, 3))
        sns.histplot(x=values, bins=30, ax=ax)
        ax.set(title=column, xlabel=column, ylabel="Count")
        fig.tight_layout()
        plt.show()

In [ ]:
# Top 10 values for up to three categorical columns, including missing values.
categorical = df.select_dtypes(include=["string", "object", "category", "bool"])
for column in categorical.columns[:3]:
    print(column)
    display(df[column].value_counts(dropna=False).head(10).to_frame("count"))

### Next steps

Choose columns from `df.columns`: compare groups, inspect a time trend, or plot two numeric variables. Adapt the commented examples below; use the metadata to decide which comparisons make sense.


In [ ]:
# Group summary (replace column names):
# df.groupby("group", dropna=False)["value"].agg(["count", "mean", "median"])

# Parse a date explicitly, then inspect a time trend:
# dated = df.assign(date=pd.to_datetime(df["date"], format="%Y-%m-%d", errors="raise"))
# dated.groupby("date")["value"].mean().plot(ylabel="Mean value")

# Relationship between two numeric variables:
# sns.scatterplot(data=df, x="x_column", y="y_column", alpha=0.5)

**Questions about the data?** opendata@fr.ch &#124; Centre de compétences et de coordination OGD / Kompetenz- und Koordinationszentrum für OGD
